#  Data Preparation

In [1]:
import pandas as pd

In [2]:
bsp_cpi = pd.read_csv('..//data//raw//bsp_cpibase2018_dataset.csv')
usd_php = pd.read_csv('..//data//raw//usd_to_php.csv')
unemployment = pd.read_csv('..//data//raw//psa_unemployment_rate.csv')
crude = pd.read_csv('..//data//raw//global_dubai_crude.csv')

In [3]:
# CPI / Inflation
bsp_cpi['Date'] = pd.to_datetime(bsp_cpi['Year'].astype(str) + ' ' + bsp_cpi['Month'] + ' 1')
bsp_cpi.rename(columns={'All Items': 'CPI'}, inplace=True)
bsp_cpi = bsp_cpi[['Date', 'CPI']].sort_values('Date')
bsp_cpi['InflationRate'] = bsp_cpi['CPI'].pct_change(12) * 100
bsp_cpi = bsp_cpi[['Date', 'InflationRate']]

# USD/PHP
usd_php['Year'] = usd_php['Period'].ffill()
usd_php['Date'] = pd.to_datetime(usd_php['Year'].astype(int).astype(str) + ' ' + usd_php['Unnamed: 1'] + ' 1')
usd_php = usd_php[['Date', 'Average']].rename(columns={'Average': 'USD_PHP'})

# Unemployment
unemployment['Date'] = pd.to_datetime(unemployment['Year'].astype(str) + ' ' + unemployment['Monthly/Quarterly'] + ' 1')
unemployment['Unemployment Rate'] = pd.to_numeric(unemployment['Unemployment Rate'], errors='coerce')
unemployment = unemployment[['Date', 'Unemployment Rate']].rename(columns={'Unemployment Rate': 'UnemploymentRate'}).sort_values('Date')
unemployment['UnemploymentRate'] = unemployment['UnemploymentRate'].interpolate(method='pad', limit_area='inside')

# Dubai crude
crude['Date'] = pd.to_datetime(crude['observation_date'], format='%d/%m/%Y')
crude = crude[['Date', 'POILDUBUSDM']].rename(columns={'POILDUBUSDM': 'DubaiCrude'})

# Merge
inf_data = bsp_cpi.merge(usd_php, on='Date', how='outer').merge(unemployment, on='Date', how='outer').merge(crude, on='Date', how='outer').sort_values('Date').reset_index(drop=True)
inf_data.set_index('Date', inplace=True)

start = inf_data['InflationRate'].first_valid_index()
end = inf_data.dropna(how='all').index[-1]
inf_data = inf_data.loc[start:end]

inf_data.to_csv('..//data//processed//combined_macro_data.csv')

C:\Users\PC\AppData\Local\Temp/ipykernel_10460/1926443933.py:17: FutureWarning: Series.interpolate with method=pad is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  unemployment['UnemploymentRate'] = unemployment['UnemploymentRate'].interpolate(method='pad', limit_area='inside')


In [4]:
inf_data.tail()

,InflationRate,USD_PHP,UnemploymentRate,DubaiCrude
Date,,,,
2026-01-01,2.024922,59.162190,5.8,62.728182
2026-02-01,2.419984,58.280263,5.1,68.508500
2026-03-01,4.068858,59.406905,5.0,126.706364
2026-04-01,7.227023,60.291316,4.7,105.299091
2026-05-01,6.761006,61.441000,4.8,102.282308
